In [1]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/pratham/datasets/chest_xray_pneumonia/chest_xray"
train_dir = f"{base_dir}/train"
test_dir  = f"{base_dir}/test"

Mounted at /content/drive


In [2]:
!mkdir -p /content/chest_xray
!cp -r "{train_dir}" /content/chest_xray/train
!cp -r "{test_dir}" /content/chest_xray/test

train_dir = "/content/chest_xray/train"
test_dir = "/content/chest_xray/test"

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
    # NO rescale here
)

test_datagen = ImageDataGenerator()  # NO rescale here either

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(224,224), batch_size=32,
    class_mode='binary', subset='training'
)

val_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(224,224), batch_size=32,
    class_mode='binary', subset='validation'
)

test_gen = test_datagen.flow_from_directory(
    test_dir, target_size=(224,224), batch_size=32,
    class_mode='binary', shuffle=False
)

Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [4]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = train_gen.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)
print(train_gen.class_indices)

{0: np.float64(1.9445479962721341), 1: np.float64(0.6730645161290323)}
{'NORMAL': 0, 'PNEUMONIA': 1}


In [5]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Epoch 1/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 171s 1s/step - accuracy: 0.8838 - loss: 0.2545 - val_accuracy: 0.8965 - val_loss: 0.2428 - learning_rate: 0.0010
Epoch 2/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 780ms/step - accuracy: 0.9149 - loss: 0.1945 - val_accuracy: 0.9051 - val_loss: 0.2229 - learning_rate: 0.0010
Epoch 3/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 776ms/step - accuracy: 0.9305 - loss: 0.1725 - val_accuracy: 0.9386 - val_loss: 0.1643 - learning_rate: 0.0010
Epoch 4/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 771ms/step - accuracy: 0.9281 - loss: 0.1632 - val_accuracy: 0.9492 - val_loss: 0.1184 - learning_rate: 0.0010
Epoch 5/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 777ms/step - accuracy: 0.9245 - loss: 0.1692 - val_accuracy: 0.9569 - val_loss: 0.1038 - learning_rate: 0.0010
Epoch 6/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 774ms/step - accuracy: 0.9363 - loss: 0.1597 - val_accuracy: 0.9501 - val_loss: 0.1192 - learning_rate: 0.0010
Epoch 7/15
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 777ms/step - accuracy:

In [7]:
loss, acc = model.evaluate(test_gen)
print("Test Accuracy (frozen base):", acc)

20/20 ━━━━━━━━━━━━━━━━━━━━ 17s 856ms/step - accuracy: 0.8942 - loss: 0.2551
Test Accuracy (frozen base): 0.8942307829856873


In [8]:
model.save(
    "/content/drive/MyDrive/pratham/models/task10_efficientnetb0_baseline.keras"
)

In [9]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 161s 990ms/step - accuracy: 0.8893 - loss: 0.2299 - val_accuracy: 0.9022 - val_loss: 0.2402 - learning_rate: 1.0000e-05
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 771ms/step - accuracy: 0.8989 - loss: 0.2086 - val_accuracy: 0.9233 - val_loss: 0.1954 - learning_rate: 1.0000e-05
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 104s 793ms/step - accuracy: 0.9128 - loss: 0.1855 - val_accuracy: 0.9233 - val_loss: 0.1871 - learning_rate: 5.0000e-06
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 774ms/step - accuracy: 0.9161 - loss: 0.1781 - val_accuracy: 0.9262 - val_loss: 0.1744 - learning_rate: 5.0000e-06


In [10]:
loss, acc = model.evaluate(test_gen)
print("Fine-tuned Test Accuracy:", acc)

model.save("/content/drive/MyDrive/pratham/models/task10_efficientnetb0_pneumonia_v2.keras")

20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 510ms/step - accuracy: 0.8718 - loss: 0.2928
Fine-tuned Test Accuracy: 0.8717948794364929
